# 🔌 Utility Asset Detector

**One-click setup for detecting T&D utility infrastructure.**

Detects: Utility poles, transmission towers, insulators, transformers, damage, etc.

**Instructions:** Click `Runtime → Run all` and wait ~5 minutes.

---

In [ ]:
#@title 1️⃣ Install & Setup (restart required after this cell)
import os

# Fix numpy version FIRST (before any other imports)
!pip uninstall -y numpy
!pip install "numpy<2.0" --quiet

# Install other deps
!pip install -q gradio pillow opencv-python pyyaml huggingface_hub

# Clone and install DART
if not os.path.exists("/content/DART"):
    !git clone https://github.com/mkturkcan/DART.git /content/DART
!pip install -e /content/DART --quiet

# Clone utility-asset-detector  
if not os.path.exists("/content/utility-asset-detector"):
    !git clone https://github.com/menonpg/utility-asset-detector.git /content/utility-asset-detector

print("\n" + "="*50)
print("⚠️  RESTART REQUIRED!")
print("   Click: Runtime → Restart session")
print("   Then run cells 2-5 (skip this cell)")
print("="*50)

In [ ]:
#@title 2️⃣ Check GPU & Imports
!nvidia-smi --query-gpu=name,memory.total --format=csv

import sys
sys.path.insert(0, "/content/DART")
sys.path.insert(0, "/content/utility-asset-detector")

import torch
import numpy as np
import sam3

print(f"\n✅ PyTorch CUDA: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")
print(f"✅ NumPy: {np.__version__}")
print(f"✅ sam3: loaded")

In [ ]:
#@title 3️⃣ Download SAM3 Checkpoint (~1.7 GB, ~2 min)
from huggingface_hub import hf_hub_download
import os

os.chdir("/content/utility-asset-detector")

if not os.path.exists("sam3.pt"):
    print("⏳ Downloading SAM3 checkpoint...")
    hf_hub_download(repo_id="bodhicitta/sam3", filename="sam3.pt", local_dir=".")
    print("✅ Downloaded!")
else:
    print("✅ Checkpoint exists")

print(f"   Size: {os.path.getsize('sam3.pt') / (1024**3):.2f} GB")

In [ ]:
#@title 4️⃣ Test Detection
import sys, os
sys.path.insert(0, "/content/DART")
sys.path.insert(0, "/content/utility-asset-detector")
os.chdir("/content/utility-asset-detector")

from PIL import Image
import requests
from io import BytesIO

# Get test image
print("Getting test image...")
try:
    r = requests.get("https://images.unsplash.com/photo-1473341304170-971dccb5ac1e?w=400", 
                     headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    img = Image.open(BytesIO(r.content))
except:
    img = Image.new('RGB', (400, 600), 'gray')
img.save("test.jpg")

# Load detector
print("Loading detector (30-60s first time)...")
from src.detector import UtilityAssetDetector
detector = UtilityAssetDetector(config="configs/assets.yaml", device="cuda")

# Test
result = detector.detect("test.jpg")
print(f"\n✅ Ready! Found {result.total_structures} structures, {result.total_components} components")

In [ ]:
#@title 5️⃣ Launch Web UI 🚀
import sys, os
sys.path.insert(0, "/content/DART")
sys.path.insert(0, "/content/utility-asset-detector")
os.chdir("/content/utility-asset-detector")

from app import create_ui
app = create_ui()
app.launch(share=True)

---
## 📝 Upload Your Own Image

In [ ]:
#@title Upload & Detect
import sys
sys.path.insert(0, "/content/DART")
sys.path.insert(0, "/content/utility-asset-detector")

from google.colab import files
from src.detector import UtilityAssetDetector

uploaded = files.upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    detector = UtilityAssetDetector(config="configs/assets.yaml", device="cuda")
    result = detector.detect(fn)
    
    print(f"\n📊 {result.total_structures} structures, {result.total_components} components, priority: {result.priority}")
    for s in result.structures:
        print(f"\n📍 {s.type} ({s.confidence:.0%}) - {s.condition.status}")
        for c in s.components:
            print(f"   • {c.type}: {c.condition.status}")